In [131]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display


In [132]:
sample_catalog_df = pd.read_csv('sample_sweetcat.csv')
best_rm_path = Path('observations/Best_RM')


def normalize_star_name(star_name):
    return re.sub(r'[^a-z0-9]', '', str(star_name).lower())


observations_df = pd.DataFrame(
    [
        {
            'observation_file': obs_path.name,
            'star_key': normalize_star_name(
                re.sub(r'_esp.*$', '', obs_path.stem, flags=re.IGNORECASE)
            ),
        }
        for obs_path in sorted(best_rm_path.glob('*.rdb'))
    ]
)
sample_catalog_with_keys = sample_catalog_df.assign(
    star_key=sample_catalog_df['star'].map(normalize_star_name)
)

sample_df = observations_df.merge(
    sample_catalog_with_keys,
    on='star_key',
    how='left',
).drop(columns='star_key')
sample_df = sample_df[['observation_file', *sample_catalog_df.columns]]

sample_df.head(2)

,observation_file,star,Name,gaia_dr3,Teff,eTeff,Logg,eLogg,[Fe/H],e[Fe/H],...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,HD189733_esp19_3.rdb,HD189733,HD 189733,1827242816201846144,4969.0,48.0,4.3,0.12,-0.08,0.03,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,HD189733_esp19_4.rdb,HD189733,HD 189733,1827242816201846144,4969.0,48.0,4.3,0.12,-0.08,0.03,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA


In [133]:
file_path = best_rm_path


def read_rdb_file(file_path):
    rdb_df = pd.read_csv(file_path, sep='\t', skiprows=[1])

    for column in ['rjd', 'vrad', 'svrad']:
        rdb_df[column] = pd.to_numeric(rdb_df[column], errors='coerce')

    return rdb_df


def plot_vrad_vs_rjd(obs_name):
    obs_path = file_path / obs_name
    rdb_df = read_rdb_file(obs_path)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=rdb_df['rjd'],
            y=rdb_df['vrad'],
            error_y=dict(type='data', array=rdb_df['svrad'], visible=True),
            mode='markers+lines',
            marker=dict(size=7),
            name=obs_name,
            text=rdb_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>'
        )
    )

    fig.update_layout(
        title=f'vrad vs rjd: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest'
    )

    fig.show()

In [134]:
all_obs_names = sorted(path.name for path in file_path.glob('*.rdb'))
obs_selector = widgets.Dropdown(options=all_obs_names, description='obs_name:')
interactive_plot = widgets.interactive_output(plot_vrad_vs_rjd, {'obs_name': obs_selector})
display(obs_selector, interactive_plot)

Dropdown(description='obs_name:', options=('HD189733_esp19_3.rdb', 'HD189733_esp19_4.rdb', 'HD209458_esp19_1.r…

Output()

In [135]:
#obs_name = 'WASP62_esp19_1.rdb'


def fit_linear_rv(obs_df, fit_mask=None):
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy()
    if fit_mask is None:
        fit_mask = pd.Series(True, index=clean_df.index)
    else:
        fit_mask = pd.Series(fit_mask, index=clean_df.index).fillna(False)

    fit_df = clean_df.loc[fit_mask]
    rjd_ref = fit_df['rjd'].median()
    x_fit = fit_df['rjd'] - rjd_ref
    weights = 1 / fit_df['svrad']
    slope, intercept = np.polyfit(x_fit, fit_df['vrad'], deg=1, w=weights)

    model = intercept + slope * (clean_df['rjd'] - rjd_ref)
    result_df = clean_df.assign(
        fit_mask=fit_mask,
        linear_model=model,
        linear_residual=clean_df['vrad'] - model,
    )
    fit_result = {
        'rjd_ref': rjd_ref,
        'intercept': intercept,
        'slope': slope,
        'n_fit_points': int(fit_mask.sum()),
    }

    return result_df, fit_result


def iteratively_exclude_largest_residuals(obs_df, n_iterations=1, n_exclude_each=1):
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy()
    fit_mask = pd.Series(True, index=clean_df.index)
    history = []

    for iteration in range(n_iterations):
        fitted_df, fit_result = fit_linear_rv(clean_df, fit_mask=fit_mask)
        candidate_residuals = fitted_df.loc[fit_mask, 'linear_residual'].abs()
        exclude_indices = candidate_residuals.nlargest(n_exclude_each).index
        fit_mask.loc[exclude_indices] = False
        history.append(
            {
                'iteration': iteration + 1,
                **fit_result,
                'excluded_indices': list(exclude_indices),
                'excluded_files': fitted_df.loc[exclude_indices, 'file_rootpath'].tolist(),
                'excluded_rjd': fitted_df.loc[exclude_indices, 'rjd'].tolist(),
                'excluded_vrad': fitted_df.loc[exclude_indices, 'vrad'].tolist(),
                'excluded_residual': fitted_df.loc[exclude_indices, 'linear_residual'].tolist(),
            }
        )

    initial_df, initial_fit = fit_linear_rv(clean_df)
    final_df, final_fit = fit_linear_rv(clean_df, fit_mask=fit_mask)

    return initial_df, initial_fit, final_df, final_fit, pd.DataFrame(history)


def model_curve(obs_df, fit_result, n_points=300):
    model_rjd = np.linspace(obs_df['rjd'].min(), obs_df['rjd'].max(), n_points)
    model_vrad = fit_result['intercept'] + fit_result['slope'] * (model_rjd - fit_result['rjd_ref'])

    return model_rjd, model_vrad


def plot_iterative_linear_fit(obs_df, initial_fit, final_df, final_fit, obs_name):
    initial_model_rjd, initial_model_vrad = model_curve(obs_df, initial_fit)
    final_model_rjd, final_model_vrad = model_curve(obs_df, final_fit)
    kept_df = final_df.loc[final_df['fit_mask']]
    excluded_df = final_df.loc[~final_df['fit_mask']]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=kept_df['rjd'],
            y=kept_df['vrad'],
            error_y=dict(type='data', array=kept_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=7),
            name='Used RV points',
            text=kept_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=excluded_df['rjd'],
            y=excluded_df['vrad'],
            error_y=dict(type='data', array=excluded_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=10, symbol='x'),
            name='Excluded largest residuals',
            text=excluded_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=initial_model_rjd,
            y=initial_model_vrad,
            mode='lines',
            line=dict(width=2, dash='dash'),
            name='Initial linear fit',
            hovertemplate='rjd=%{x}<br>initial model=%{y}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=final_model_rjd,
            y=final_model_vrad,
            mode='lines',
            line=dict(width=3),
            name='Refit after exclusions',
            hovertemplate='rjd=%{x}<br>refit model=%{y}<extra></extra>',
        )
    )

    fig.update_layout(
        title=f'Iterative linear RV fit: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest',
    )

    return fig


"""
obs_df = read_rdb_file(best_rm_path / obs_name)
initial_df, initial_fit, final_df, final_fit, exclusion_history = iteratively_exclude_largest_residuals(
    obs_df,
    n_iterations=12,
    n_exclude_each=1,
)

print(
    f"initial fit: vrad = {initial_fit['intercept']:.6f} "
    f"+ {initial_fit['slope']:.6f} * (rjd - {initial_fit['rjd_ref']:.8f})"
)
print(
    f"refit: vrad = {final_fit['intercept']:.6f} "
    f"+ {final_fit['slope']:.6f} * (rjd - {final_fit['rjd_ref']:.8f})"
)
display(exclusion_history)

fig = plot_iterative_linear_fit(obs_df, initial_fit, final_df, final_fit, obs_name)
fig.show()
"""

'\nobs_df = read_rdb_file(best_rm_path / obs_name)\ninitial_df, initial_fit, final_df, final_fit, exclusion_history = iteratively_exclude_largest_residuals(\n    obs_df,\n    n_iterations=12,\n    n_exclude_each=1,\n)\n\nprint(\n    f"initial fit: vrad = {initial_fit[\'intercept\']:.6f} "\n    f"+ {initial_fit[\'slope\']:.6f} * (rjd - {initial_fit[\'rjd_ref\']:.8f})"\n)\nprint(\n    f"refit: vrad = {final_fit[\'intercept\']:.6f} "\n    f"+ {final_fit[\'slope\']:.6f} * (rjd - {final_fit[\'rjd_ref\']:.8f})"\n)\ndisplay(exclusion_history)\n\nfig = plot_iterative_linear_fit(obs_df, initial_fit, final_df, final_fit, obs_name)\nfig.show()\n'

In [136]:
obs_name = 'WASP131_esp19_1.rdb'
obs_df = read_rdb_file(best_rm_path / obs_name)

def iteratively_clip_by_rms(obs_name, rms_threshold=2.0, max_iterations=10, observations_path=best_rm_path):
    obs_df = read_rdb_file(observations_path / obs_name)
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy()
    fit_mask = pd.Series(True, index=clean_df.index)
    history = []

    for iteration in range(max_iterations):
        fitted_df, fit_result = fit_linear_rv(clean_df, fit_mask=fit_mask)
        fit_residuals = fitted_df.loc[fit_mask, 'linear_residual']
        residual_rms = np.sqrt(np.mean(fit_residuals**2))
        clip_limit = rms_threshold * residual_rms
        exclude_mask = fit_mask & (fitted_df['linear_residual'].abs() > clip_limit)
        exclude_indices = fitted_df.index[exclude_mask]

        history.append(
            {
                'iteration': iteration + 1,
                **fit_result,
                'residual_rms': residual_rms,
                'clip_limit': clip_limit,
                'n_excluded': len(exclude_indices),
                'excluded_indices': list(exclude_indices),
                'excluded_files': fitted_df.loc[exclude_indices, 'file_rootpath'].tolist(),
                'excluded_rjd': fitted_df.loc[exclude_indices, 'rjd'].tolist(),
                'excluded_vrad': fitted_df.loc[exclude_indices, 'vrad'].tolist(),
                'excluded_residual': fitted_df.loc[exclude_indices, 'linear_residual'].tolist(),
            }
        )

        if len(exclude_indices) == 0:
            break

        fit_mask.loc[exclude_indices] = False

    initial_df, initial_fit = fit_linear_rv(clean_df)
    final_df, final_fit = fit_linear_rv(clean_df, fit_mask=fit_mask)

    return initial_df, initial_fit, final_df, final_fit, pd.DataFrame(history)


def fit_with_edge_points(obs_name, n_first=5, n_last=6, observations_path=best_rm_path):
    obs_df = read_rdb_file(observations_path / obs_name)
    clean_df = obs_df.dropna(subset=['rjd', 'vrad', 'svrad']).copy().sort_values('rjd')

    fit_mask = pd.Series(False, index=clean_df.index)
    fit_mask.loc[clean_df.head(n_first).index] = True
    fit_mask.loc[clean_df.tail(n_last).index] = True

    fitted_df, fit_result = fit_linear_rv(clean_df, fit_mask=fit_mask)
    fit_history = pd.DataFrame(
        [
            {
                **fit_result,
                'fit_method': 'edge_points',
                'n_first': n_first,
                'n_last': n_last,
                'fit_indices': list(clean_df.index[fit_mask]),
            }
        ]
    )

    return fitted_df, fit_result, fit_history


def plot_linear_fit_selection(obs_df, fitted_df, fit_result, obs_name, fit_label='Linear fit'):
    model_rjd, model_vrad = model_curve(obs_df, fit_result)
    fit_df = fitted_df.loc[fitted_df['fit_mask']]
    unused_df = fitted_df.loc[~fitted_df['fit_mask']]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=unused_df['rjd'],
            y=unused_df['vrad'],
            error_y=dict(type='data', array=unused_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=7, opacity=0.55),
            name='Not used in fit',
            text=unused_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=fit_df['rjd'],
            y=fit_df['vrad'],
            error_y=dict(type='data', array=fit_df['svrad'], visible=True),
            mode='markers',
            marker=dict(size=10, symbol='circle-open'),
            name='Used in fit',
            text=fit_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>',
        )
    )
    fig.add_trace(
        go.Scatter(
            x=model_rjd,
            y=model_vrad,
            mode='lines',
            line=dict(width=3),
            name=fit_label,
            hovertemplate='rjd=%{x}<br>model vrad=%{y}<extra></extra>',
        )
    )
    fig.update_layout(
        title=f'{fit_label}: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest',
    )

    return fig


fit_method_for_plot = 'edge_points'  # use 'rms_clip' or 'edge_points'

if fit_method_for_plot == 'rms_clip':
    rms_initial_df, rms_initial_fit, rms_final_df, rms_final_fit, rms_clip_history = iteratively_clip_by_rms(
        obs_name,
        rms_threshold=1.5,
        max_iterations=5,
    )
    print(
        f"RMS-clipped fit: vrad = {rms_final_fit['intercept']:.6f} "
        f"+ {rms_final_fit['slope']:.6f} * (rjd - {rms_final_fit['rjd_ref']:.8f})"
    )
    fig = plot_iterative_linear_fit(obs_df, rms_initial_fit, rms_final_df, rms_final_fit, obs_name)
    fig.update_layout(title=f'RMS-clipped linear RV fit: {obs_name}')
elif fit_method_for_plot == 'edge_points':
    edge_fitted_df, edge_fit, edge_fit_history = fit_with_edge_points(
        obs_name,
        n_first=20,
        n_last=20,
    )
    print(
        f"edge-points fit: vrad = {edge_fit['intercept']:.6f} "
        f"+ {edge_fit['slope']:.6f} * (rjd - {edge_fit['rjd_ref']:.8f})"
    )
    display(edge_fit_history)
    fig = plot_linear_fit_selection(obs_df, edge_fitted_df, edge_fit, obs_name, fit_label='Edge-points linear fit')
else:
    raise ValueError("fit_method_for_plot must be 'rms_clip' or 'edge_points'")

fig.show()

edge-points fit: vrad = -19718.270001 + -21.749580 * (rjd - 59282.77265872)


,rjd_ref,intercept,slope,n_fit_points,fit_method,n_first,n_last,fit_indices
0,59282.772659,-19718.270001,-21.74958,40,edge_points,20,20,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."


In [137]:
def make_linear_corrected_observation_df(
    obs_name,
    fit_method='rms_clip',
    rms_threshold=1.5,
    max_iterations=7,
    n_first=6,
    n_last=5,
    observations_path=best_rm_path,
):
    if fit_method == 'rms_clip':
        _, _, fitted_df, fit_result, fit_history = iteratively_clip_by_rms(
            obs_name,
            rms_threshold=rms_threshold,
            max_iterations=max_iterations,
            observations_path=observations_path,
        )
    elif fit_method == 'edge_points':
        fitted_df, fit_result, fit_history = fit_with_edge_points(
            obs_name,
            n_first=n_first,
            n_last=n_last,
            observations_path=observations_path,
        )
    else:
        raise ValueError("fit_method must be 'rms_clip' or 'edge_points'")

    output_columns = ['rjd', 'vrad', 'fwhm', 'bis_span', 'contrast', 's_mw', 'ha', 'na', 'ca', 'rhk']
    for column in output_columns:
        if column not in fitted_df.columns:
            fitted_df[column] = np.nan

    corrected_df = fitted_df[output_columns].copy()
    corrected_df['true_vrad'] = fitted_df['vrad'] - fitted_df['linear_model']
    corrected_df['true_vrad'] = np.round(corrected_df['true_vrad'] - corrected_df['true_vrad'].mean(), decimals=2)
    corrected_df['true_vrad'] = corrected_df['true_vrad'] - corrected_df['true_vrad'].mean()
    corrected_df = corrected_df[
        ['rjd', 'vrad', 'true_vrad', 'fwhm', 'bis_span', 'contrast', 's_mw', 'ha', 'na', 'ca', 'rhk']
    ]

    output_path = observations_path / "linear_corrected"
    output_path.mkdir(exist_ok=True)
    corrected_df.to_csv(output_path / f"{obs_name.replace('.rdb', '')}_linear_corrected.csv", index=False)

    return corrected_df, fit_result, fit_history, fitted_df


#obs_name = 'HD209458_esp19_1.rdb'

corrected_obs_df, corrected_fit, corrected_clip_history, corrected_fit_df = make_linear_corrected_observation_df(
    obs_name,
    fit_method='rms_clip',
    rms_threshold=1.5,
    max_iterations=5,
    n_first=5,
    n_last=4,
)

print(
    f"linear correction fit: vrad = {corrected_fit['intercept']:.6f} "
    f"+ {corrected_fit['slope']:.6f} * (rjd - {corrected_fit['rjd_ref']:.8f})"
)
print(f"mean(true_vrad) = {corrected_obs_df['true_vrad'].mean():.6e}")
display(corrected_obs_df)

linear correction fit: vrad = -19718.266697 + -22.032833 * (rjd - 59282.77280130)
mean(true_vrad) = 5.730183e-17


,rjd,vrad,true_vrad,fwhm,bis_span,contrast,s_mw,ha,na,ca,rhk
0,59282.645324,-19713.799957,4.849677,8644.718925,4.816560,48.366791,154.501576,0.196948,0.386913,0.118297,-4.900226
1,59282.649434,-19715.025649,3.709677,8642.649405,7.671718,48.370917,160.638269,0.195012,0.385063,0.124175,-4.862162
2,59282.653571,-19713.366207,5.459677,8660.654306,3.819733,48.402106,162.944790,0.195531,0.384739,0.126384,-4.848676
3,59282.657688,-19714.184106,4.729677,8656.184380,12.562440,48.376263,160.558197,0.195961,0.387011,0.124098,-4.862638
4,59282.662001,-19715.321697,3.689677,8654.096021,1.700327,48.365069,160.889011,0.196045,0.385621,0.124415,-4.860676
...,...,...,...,...,...,...,...,...,...,...,...
57,59282.883527,-19724.011339,-0.120323,8639.300970,1.283560,48.415019,163.026712,0.198043,0.385182,0.126462,-4.848205
58,59282.887780,-19724.808931,-0.820323,8653.710790,4.427148,48.370778,162.272794,0.197762,0.385581,0.125740,-4.852562
59,59282.891921,-19724.766590,-0.690323,8640.186948,6.445780,48.383542,163.806357,0.197432,0.385326,0.127209,-4.843744
60,59282.896165,-19722.821729,1.349677,8651.266919,9.021353,48.345691,161.741002,0.197649,0.386161,0.125231,-4.855662


In [138]:
corrected_obs_df.describe()

,rjd,vrad,true_vrad,fwhm,bis_span,contrast,s_mw,ha,na,ca,rhk
count,62.000000,62.000000,6.200000e+01,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000
mean,59282.772736,-19721.453106,5.730183e-17,8649.099326,7.216975,48.363632,161.899227,0.196736,0.385927,0.125382,-4.854890
std,0.075456,5.445021,5.294497e+00,7.187041,4.506897,0.033772,1.957119,0.001132,0.001044,0.001875,0.011670
min,59282.645324,-19735.198095,-1.416032e+01,8625.593490,-8.587065,48.307923,154.501576,0.193518,0.383557,0.118297,-4.900226
25%,59282.708986,-19724.798346,-2.795323e+00,8644.122397,4.921863,48.341271,160.941728,0.196086,0.385322,0.124465,-4.860364
50%,59282.772768,-19720.620368,1.454677e+00,8649.400921,7.258166,48.361077,162.520677,0.196549,0.385939,0.125977,-4.851125
75%,59282.836485,-19717.456156,4.059677e+00,8654.161907,10.073024,48.386017,163.044206,0.197642,0.386663,0.126479,-4.848104
max,59282.900289,-19713.229998,7.159677e+00,8665.955120,16.782048,48.454909,164.426905,0.198789,0.388132,0.127804,-4.840226
